# Synthetic CDV Experiment - Analysis

This notebook analyzes the multi-seed experiment results across 4 alpha levels.

**Key Questions:**
1. Does CDV matching converge to global performance at alpha=0 (homogeneous)?
2. Does CDV outperform global under heterogeneity (alpha > 0)?
3. How does the advantage scale with alpha?
4. Is the difference statistically significant?

In [ ]:
import os
import json as json_lib
import pickle
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

from cdv_utils.experiment_runner import load_experiment_results
from cdv_utils.results_analysis import (
    prepare_dataframes_for_analysis,
    calculate_ate_mse_decomposition,
    calculate_ate_by_estimator,
    calculate_ite_mse_pehe,
    perform_ate_mse_statistical_tests,
    perform_cate_mse_statistical_tests,
    add_error_columns,
)

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("colorblind")

# Load config
with open("results/synthetic_experiment_config.json", "r") as f:
    config = json_lib.load(f)

ALPHA_VALUES = config["alpha_values"]
print(f"Alpha values: {ALPHA_VALUES}")
print("Analysis loaded and ready.")

## 1. Load All Results

In [ ]:
# Load results for all alpha values
results_by_alpha = {}
for alpha in ALPHA_VALUES:
    path = f"results/synthetic_alpha_{alpha:.2f}.pkl"
    if os.path.exists(path):
        results_by_alpha[alpha] = load_experiment_results(path)
        print(f"alpha={alpha:.2f}: loaded {len(results_by_alpha[alpha])} seeds")
    else:
        print(f"alpha={alpha:.2f}: NOT FOUND - run experiment first!")

print(f"\nLoaded {len(results_by_alpha)} alpha levels.")

# Quick sanity check: inspect the structure of one seed result
sample_alpha = ALPHA_VALUES[0]
sample_seed = list(results_by_alpha[sample_alpha].keys())[0]
sample_result = results_by_alpha[sample_alpha][sample_seed]
print(f"\nResult keys per seed: {list(sample_result.keys())}")
for key, val in sample_result.items():
    if isinstance(val, pd.DataFrame):
        print(f"  {key}: DataFrame {val.shape}, columns={list(val.columns)}")
    else:
        print(f"  {key}: {type(val).__name__}")

## 2. Prepare DataFrames per Alpha

Using the same `prepare_dataframes_for_analysis()` function as in Notebook 03, 
we extract and concatenate the per-seed result DataFrames into unified DataFrames 
for each alpha level.


In [ ]:
# Prepare analysis dataframes for each alpha level
analysis_by_alpha = {}

for alpha in ALPHA_VALUES:
    print(f"\n{'='*70}")
    print(f"ALPHA = {alpha}")
    print(f"{'='*70}")
    
    results_by_seed = results_by_alpha[alpha]
    seeds = sorted(results_by_seed.keys())
    
    dataframes = prepare_dataframes_for_analysis(results_by_seed, seeds)
    analysis_by_alpha[alpha] = {
        'dataframes': dataframes,
        'seeds': seeds,
    }

print(f"\nPrepared analysis for {len(analysis_by_alpha)} alpha levels.")

## 3. ATE MSE Decomposition per Alpha


In [ ]:
# ATE-level MSE decomposition for each alpha
ate_decomposition_by_alpha = {}
ate_by_seed_by_alpha = {}

for alpha in ALPHA_VALUES:
    print(f"\n--- Alpha = {alpha} ---")
    dfs = analysis_by_alpha[alpha]['dataframes']
    seeds = analysis_by_alpha[alpha]['seeds']
    
    decomp_df, ate_by_seed_df = calculate_ate_mse_decomposition(
        dfs['DF_BEST_GLOBAL'], dfs['DF_BEST_VARIANT'], seeds
    )
    
    ate_decomposition_by_alpha[alpha] = decomp_df
    ate_by_seed_by_alpha[alpha] = ate_by_seed_df

# Summary across all alpha values
print("\n" + "=" * 80)
print("ATE MSE DECOMPOSITION SUMMARY")
print("=" * 80)
for alpha in ALPHA_VALUES:
    decomp = ate_decomposition_by_alpha[alpha]
    for _, row in decomp.iterrows():
        print(f"  alpha={alpha:.2f} | {row['method']:>8}: "
              f"Bias²={row['bias_squared']:.6f}, Var={row['variance']:.6f}, MSE={row['mse']:.6f}")

## 4. CATE (ITE) MSE per Alpha


In [ ]:
# CATE (ITE-level) MSE/PEHE per alpha
cate_summary_by_alpha = {}
cate_decomposition_by_alpha = {}

for alpha in ALPHA_VALUES:
    print(f"\n--- Alpha = {alpha} ---")
    dfs = analysis_by_alpha[alpha]['dataframes']
    
    ite_summary_df, ite_decomposition_df = calculate_ite_mse_pehe(
        dfs['DF_ALL_GLOBAL'], dfs['DF_ALL_VARIANT']
    )
    cate_summary_by_alpha[alpha] = ite_summary_df
    cate_decomposition_by_alpha[alpha] = ite_decomposition_df

# Summary
print("\n" + "=" * 80)
print("CATE (ITE) MSE DECOMPOSITION SUMMARY - BEST ESTIMATOR PER METHOD")
print("=" * 80)
for alpha in ALPHA_VALUES:
    decomp = cate_decomposition_by_alpha[alpha]
    # Show the best estimator per method (lowest MSE)
    for method in ['global', 'variant']:
        method_rows = decomp[decomp['method'] == method]
        if len(method_rows) > 0:
            best = method_rows.loc[method_rows['mse_empirical'].idxmin()]
            print(f"  alpha={alpha:.2f} | {method:>8} | {best['estimator']}: "
                  f"Bias²={best['bias_squared']:.4f}, Var={best['variance']:.4f}, "
                  f"MSE={best['mse_empirical']:.4f}")

## 5. Statistical Significance Tests per Alpha


In [ ]:
# ATE statistical tests per alpha
ate_stat_by_alpha = {}

for alpha in ALPHA_VALUES:
    print(f"\n{'='*70}")
    print(f"ALPHA = {alpha}")
    print(f"{'='*70}")
    dfs = analysis_by_alpha[alpha]['dataframes']
    seeds = analysis_by_alpha[alpha]['seeds']
    
    # Need per-estimator ATE results for the statistical test function
    estimator_ate_summary, estimator_ate_by_seed = calculate_ate_by_estimator(
        dfs['DF_ALL_GLOBAL'], dfs['DF_ALL_VARIANT'], seeds
    )
    
    ate_stat_df = perform_ate_mse_statistical_tests(
        estimator_ate_by_seed, estimator_ate_summary,
        df_best_global=dfs['DF_BEST_GLOBAL'],
        df_best_variant=dfs['DF_BEST_VARIANT']
    )
    ate_stat_by_alpha[alpha] = ate_stat_df

## 6. CATE Statistical Tests per Alpha


In [ ]:
# CATE statistical tests per alpha
cate_stat_by_alpha = {}

for alpha in ALPHA_VALUES:
    print(f"\n{'='*70}")
    print(f"ALPHA = {alpha}")
    print(f"{'='*70}")
    dfs = analysis_by_alpha[alpha]['dataframes']
    
    cate_stat_df = perform_cate_mse_statistical_tests(
        dfs['DF_ALL_GLOBAL'], dfs['DF_ALL_VARIANT'],
        df_best_global=dfs['DF_BEST_GLOBAL'],
        df_best_variant=dfs['DF_BEST_VARIANT']
    )
    cate_stat_by_alpha[alpha] = cate_stat_df

## 7. Scissors Chart (ATE MSE vs Alpha)


In [ ]:
# Build scissors data from ATE decomposition results
alphas = sorted(ALPHA_VALUES)

cdv_ate_mse = []
global_ate_mse = []
cdv_ate_bias2 = []
global_ate_bias2 = []
cdv_ate_var = []
global_ate_var = []

for alpha in alphas:
    decomp = ate_decomposition_by_alpha[alpha]
    global_row = decomp[decomp['method'] == 'global'].iloc[0]
    variant_row = decomp[decomp['method'] == 'variant'].iloc[0]
    
    global_ate_mse.append(global_row['mse'])
    cdv_ate_mse.append(variant_row['mse'])
    global_ate_bias2.append(global_row['bias_squared'])
    cdv_ate_bias2.append(variant_row['bias_squared'])
    global_ate_var.append(global_row['variance'])
    cdv_ate_var.append(variant_row['variance'])

# Scissors chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: ATE MSE
axes[0].plot(alphas, cdv_ate_mse, "o-", label="CDV", color="tab:blue", linewidth=2, markersize=8)
axes[0].plot(alphas, global_ate_mse, "s--", label="Global", color="tab:red", linewidth=2, markersize=8)
axes[0].set_xlabel("Alpha (Heterogeneity Level)", fontsize=12)
axes[0].set_ylabel("ATE MSE", fontsize=12)
axes[0].set_title("ATE Estimation Error", fontsize=13)
axes[0].legend(fontsize=11)
axes[0].set_xticks(alphas)

# Panel 2: ATE Bias² component
axes[1].plot(alphas, cdv_ate_bias2, "o-", label="CDV Bias²", color="tab:blue", linewidth=2, markersize=8)
axes[1].plot(alphas, global_ate_bias2, "s--", label="Global Bias²", color="tab:red", linewidth=2, markersize=8)
axes[1].set_xlabel("Alpha (Heterogeneity Level)", fontsize=12)
axes[1].set_ylabel("ATE Bias²", fontsize=12)
axes[1].set_title("ATE Bias² Component", fontsize=13)
axes[1].legend(fontsize=11)
axes[1].set_xticks(alphas)

plt.tight_layout()
os.makedirs("plots", exist_ok=True)
plt.savefig("plots/scissors_chart_ate_synthetic.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: plots/scissors_chart_ate_synthetic.png")

## 8. Bias-Variance Decomposition Bars


In [ ]:
# Bias-Variance decomposition bar chart (ATE level)
fig, axes = plt.subplots(1, len(ALPHA_VALUES), figsize=(4*len(ALPHA_VALUES), 5), sharey=True)

for i, alpha in enumerate(alphas):
    x = [0, 1]
    bias2 = [cdv_ate_bias2[i], global_ate_bias2[i]]
    variance = [cdv_ate_var[i], global_ate_var[i]]
    
    axes[i].bar(x, bias2, width=0.4, label="Bias²", color="tab:orange", alpha=0.8)
    axes[i].bar(x, variance, width=0.4, bottom=bias2, label="Variance", color="tab:blue", alpha=0.8)
    axes[i].set_xticks(x)
    axes[i].set_xticklabels(["CDV", "Global"])
    axes[i].set_title(f"α = {alpha:.2f}", fontsize=12)
    if i == 0:
        axes[i].set_ylabel("ATE MSE Decomposition", fontsize=11)
    axes[i].legend(fontsize=9)

plt.suptitle("Bias-Variance Decomposition across Heterogeneity Levels", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("plots/bias_variance_synthetic.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: plots/bias_variance_synthetic.png")

## 9. Summary Table for Paper


In [ ]:
# Generate LaTeX-ready summary table
summary_rows = []
for alpha in alphas:
    decomp = ate_decomposition_by_alpha[alpha]
    global_row = decomp[decomp['method'] == 'global'].iloc[0]
    variant_row = decomp[decomp['method'] == 'variant'].iloc[0]
    
    row = {
        "α": f"{alpha:.2f}",
        "CDV ATE MSE": f"{variant_row['mse']:.4f}",
        "Global ATE MSE": f"{global_row['mse']:.4f}",
        "CDV Bias²": f"{variant_row['bias_squared']:.4f}",
        "Global Bias²": f"{global_row['bias_squared']:.4f}",
        "CDV Var": f"{variant_row['variance']:.4f}",
        "Global Var": f"{global_row['variance']:.4f}",
        "Seeds": f"{int(variant_row['n_seeds'])}",
    }
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
print("ATE MSE Summary Table for Paper")
print("=" * 100)
print(summary_df.to_string(index=False))
print("\n\nLaTeX format:")
print(summary_df.to_latex(index=False))

## 10. Four-Panel Comparison Figure


In [ ]:
# 4-panel figure: ATE MSE scissors + Bias² scissors + Bias-Var bars for extreme alphas + CATE decomp
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel (a): ATE MSE scissors
axes[0,0].plot(alphas, cdv_ate_mse, "o-", label="CDV", color="tab:blue", linewidth=2, markersize=8)
axes[0,0].plot(alphas, global_ate_mse, "s--", label="Global", color="tab:red", linewidth=2, markersize=8)
axes[0,0].set_xlabel("α (Heterogeneity)")
axes[0,0].set_ylabel("ATE MSE")
axes[0,0].set_title("(a) ATE MSE vs Heterogeneity")
axes[0,0].legend()
axes[0,0].set_xticks(alphas)

# Panel (b): ATE Bias² scissors
axes[0,1].plot(alphas, cdv_ate_bias2, "o-", label="CDV Bias²", color="tab:blue", linewidth=2, markersize=8)
axes[0,1].plot(alphas, global_ate_bias2, "s--", label="Global Bias²", color="tab:red", linewidth=2, markersize=8)
axes[0,1].set_xlabel("α (Heterogeneity)")
axes[0,1].set_ylabel("Bias²")
axes[0,1].set_title("(b) ATE Bias² vs Heterogeneity")
axes[0,1].legend()
axes[0,1].set_xticks(alphas)

# Panel (c): Bias-Variance bars for all alpha levels
x = np.arange(len(alphas))
width = 0.35
axes[1,0].bar(x - width/2, cdv_ate_mse, width, label="CDV", color="tab:blue", alpha=0.8)
axes[1,0].bar(x + width/2, global_ate_mse, width, label="Global", color="tab:red", alpha=0.8)
axes[1,0].set_xticks(x)
axes[1,0].set_xticklabels([f"{a:.2f}" for a in alphas])
axes[1,0].set_xlabel("α")
axes[1,0].set_ylabel("ATE MSE")
axes[1,0].set_title("(c) ATE MSE Comparison")
axes[1,0].legend()

# Panel (d): CDV improvement (%) vs alpha
improvements = []
for i in range(len(alphas)):
    if global_ate_mse[i] > 0:
        imp = (global_ate_mse[i] - cdv_ate_mse[i]) / global_ate_mse[i] * 100
    else:
        imp = 0
    improvements.append(imp)

colors_bar = ['tab:green' if imp > 0 else 'tab:red' for imp in improvements]
axes[1,1].bar(range(len(alphas)), improvements, color=colors_bar, alpha=0.8)
axes[1,1].set_xticks(range(len(alphas)))
axes[1,1].set_xticklabels([f"{a:.2f}" for a in alphas])
axes[1,1].axhline(y=0, color="black", linestyle="-", linewidth=0.5)
axes[1,1].set_xlabel("α")
axes[1,1].set_ylabel("CDV Improvement (%)")
axes[1,1].set_title("(d) CDV MSE Improvement over Global")

plt.tight_layout()
plt.savefig("plots/four_panel_synthetic.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: plots/four_panel_synthetic.png")

## 11. Key Findings

**Expected Results:**
- At α=0 (no heterogeneity): CDV and Global should produce similar ATE MSE
- As α increases: CDV should increasingly outperform Global
- The scissors chart should show diverging lines
- The improvement percentage should grow monotonically with α

**Interpretation:**
- **CDV is safe:** it does not harm performance when heterogeneity is absent (α=0)
- **CDV is beneficial:** it captures variant-specific causal structure that the global model misses
- **The advantage scales with heterogeneity:** stronger heterogeneity → larger CDV advantage
